# 02 · Тесты

Запуск тестов из ноутбука с итоговой таблицей. Каждый набор идёт **в отдельном процессе**
(как `pytest` в терминале): тесты не делят память с ядром ноутбука, а наборы под MPI
запускаются через `mpirun`.

| набор | что проверяет | DOLFINx | время |
|---|---|---|---|
| лёгкие | конфигурация, модели клетки, TNNPM против эталона, расписание, ввод-вывод, анализ, архитектура | нет | ~1 мин |
| расчётные (`-m fem`) | сетки, ткань, перенос, солверы, связанный счёт, чекпоинты, TNNPM в ткани | да | ~5 мин |
| MPI | те же расчётные на нескольких рангах | да | ~5 мин |

Флажки `RUN_*` в ячейках включают соответствующий набор.

In [ ]:
# Общая настройка: пакет cardiac_em и помощники ноутбуков доступны из любой папки проекта
import sys
from pathlib import Path

for _p in (Path.cwd(), *Path.cwd().parents):
    if (_p / "notebooks" / "nbtools.py").exists():
        sys.path.insert(0, str(_p / "notebooks"))
        break
from nbtools import ROOT, RUNS, run_stream, style  # noqa: E402

print("корень проекта:", ROOT)

## Окружение

In [ ]:
import importlib, os, shutil, platform

def version(name):
    try:
        return getattr(importlib.import_module(name), "__version__", "есть")
    except ImportError:
        return "нет"

env = {"python": platform.python_version(), "numpy": version("numpy"), "scipy": version("scipy"),
       "dolfinx": version("dolfinx"), "petsc4py": version("petsc4py"), "mpi4py": version("mpi4py"),
       "pytest": version("pytest"), "mpirun": shutil.which("mpirun") or "нет",
       "ядер CPU": os.cpu_count()}
for k, v in env.items():
    print(f"  {k:10} {v}")
HAS_DOLFINX = env["dolfinx"] != "нет"
HAS_MPI = env["mpirun"] != "нет"

In [ ]:
import pandas as pd
from nbtools import pytest_run

results = []          # сюда складываются сводки всех запусков — таблица в конце
TESTS = ROOT / "tests"

## 1. Лёгкие тесты (без DOLFINx)

Идут где угодно, в том числе на ноутбуке без расчётного стека.

In [ ]:
LIGHT = ["test_config.py", "test_architecture.py", "test_schedule.py", "test_cell_models.py",
         "test_tnnpm.py", "test_io_light.py", "test_control.py", "test_analysis.py",
         "test_queue.py"]
RUN_LIGHT = True

if RUN_LIGHT:
    results.append(pytest_run([TESTS / f for f in LIGHT], label="лёгкие"))

## 2. Расчётные тесты (DOLFINx)

Все тесты с меткой `fem`. Без DOLFINx они пропускаются, а не падают.

In [ ]:
RUN_FEM = HAS_DOLFINX

if RUN_FEM:
    results.append(pytest_run([TESTS, "-m", "fem"], label="расчётные"))
else:
    print("DOLFINx нет в этом ядре — расчётные тесты пропущены")

## 3. Под MPI

Тесты, в которых важна параллельная часть: сборка и порядок узлов, чекпоинты, общая
папка вывода, серии, TNNPM в ткани.

In [ ]:
MPI_FILES = ["test_mesh.py", "test_simulation.py", "test_io.py", "test_control_fem.py",
             "test_analysis_fem.py", "test_tnnpm_fem.py"]
N_PROC = 2
RUN_MPI = HAS_DOLFINX and HAS_MPI

if RUN_MPI:
    for f in MPI_FILES:
        results.append(pytest_run([TESTS / f], mpi=N_PROC, label=f"{f} (MPI ×{N_PROC})", tail=3))

## 4. Отдельный тест или файл

Для отладки: путь к файлу, `файл::тест` или выражение `-k`. `-x` — остановиться на первой
ошибке, `-v` — по строке на тест.

In [ ]:
TARGET = [TESTS / "test_tnnpm.py", "-v"]          # например: [TESTS / "test_io.py::test_snapshot_layout_and_content"]
RUN_TARGET = False

if RUN_TARGET:
    results.append(pytest_run(TARGET, label="выбранное", tail=60))

## Итог

In [ ]:
table = pd.DataFrame(results)
if table.empty:
    print("ничего не запускалось")
else:
    table["итог"] = ["✔ ок" if r.код == 0 else "✘ ошибки" for r in table.itertuples()]
    display(table[["итог", "набор", "passed", "failed", "errors", "skipped", "секунд"]])
    print(f"всего: прошло {table.passed.sum()}, упало {table.failed.sum()}, "
          f"ошибок {table.errors.sum()}, пропущено {table.skipped.sum()}")